In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
from pathlib import Path

# ====== CONFIG ======
RAW_PDF_DIR = Path("/content/drive/MyDrive/DSA4265_Project/Transcripts")   # your uploaded PDFs are here
PROJECT_DIR = Path("/content/drive/MyDrive/DSA4265_Project/Results")

RAW_TEXT_DIR = PROJECT_DIR / "raw_text"
CLEAN_TXT_DIR = PROJECT_DIR / "cleaned_transcripts"
SENTENCE_DIR = PROJECT_DIR / "sentence_data"
CHUNK_DIR = PROJECT_DIR / "chunk_data"
RETRIEVER_DIR = PROJECT_DIR / "retriever_data"
LLM_DIR = PROJECT_DIR / "llm_data"

for d in [RAW_TEXT_DIR, CLEAN_TXT_DIR, SENTENCE_DIR, CHUNK_DIR, RETRIEVER_DIR, LLM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Folders ready:", PROJECT_DIR)

Folders ready: /content/drive/MyDrive/DSA4265_Project/Results


In [ ]:
!pip install faiss-cpu
!pip install sentence-transformers
!pip install Qdrant-client

## building the vector index

In [1]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# 1. Load your chunks
chunks_df = pd.read_csv(CHUNK_DIR / "rag_chunks.csv")

# 2. Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Embed all chunk texts
texts = chunks_df["chunk_text"].tolist()
embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

# 4. Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
faiss.normalize_L2(embeddings)
index.add(embeddings)

# 5. Save index + metadata
faiss.write_index(index, str(RETRIEVER_DIR / "faiss.index"))
chunks_df.to_csv(RETRIEVER_DIR / "chunks_with_index.csv", index=False)

print(f"Indexed {len(texts)} chunks")

c:\Users\owenl\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

## building the retriever function

In [5]:
def retrieve(query, model, index, chunks_df, top_k=5, filter_company=None, filter_quarter=None):
    # Embed the query
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    # Search — over-fetch to allow for filtering
    scores, indices = index.search(q_emb, top_k * 3)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = chunks_df.iloc[idx]

        # Optional filters
        if filter_company and row["company"].lower() != filter_company.lower():
            continue
        if filter_quarter and row["quarter"] != filter_quarter:
            continue

        results.append({
            "chunk_id": row["chunk_id"],
            "company": row["company"],
            "quarter": row["quarter"],
            "score": float(score),
            "chunk_text": row["chunk_text"]
        })

        if len(results) == top_k:
            break

    return results

## testing the retriever function

In [ ]:
# Quick test
results = retrieve(
    query="What were the key highlights from this earnings call?",
    model=model,
    index=index,
    chunks_df=chunks_df,
    top_k=5,
    filter_company="amazon",
    filter_quarter="Q3"
)

for r in results:
    print(r["chunk_id"], "| score:", round(r["score"], 3))
    print(r["chunk_text"][:200])
    print("---")

amazon_Q3_2025_chunk_0 | score: 0.488
[Speaker: Dave Fildes] Hello, and welcome to our Q3 2025 financial results conference call. Joining us today to answer your questions is Andy Jassy, our CEO; and Brian Olsavsky, our CFO. As you listen
---


In [ ]:
print(chunks_df["company"].unique())
print(chunks_df["quarter"].unique())

['amazon' 'apple' 'meta' 'tesla']
['Q3' 'Q4']


## Building Qdrant Vector Database

In [ ]:
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# 1. Load your chunks
BASE_DIR = Path("DSA4265_RAG.ipynb").parent
print(BASE_DIR)
chunks_df = pd.read_csv(BASE_DIR / "Results" / "chunk_data" / "rag_chunks.csv")

# 2. Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Embed all chunk texts
texts = chunks_df["chunk_text"].tolist()
embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

# 4. Connect to Qdrant and create collection

qdrant_path = BASE_DIR / "Results" / "retriever_data" / "qdrant_store"
qdrant_path.mkdir(parents=True, exist_ok=True)

# Wipe any corrupted store from a previous run
if qdrant_path.exists():
    shutil.rmtree(qdrant_path)
qdrant_path.mkdir(parents=True, exist_ok=True)

client = QdrantClient(path=str(BASE_DIR / "Results" / "retriever_data" / "qdrant_store"))

if client.collection_exists(collection_name="chunks"):
    client.delete_collection(collection_name="chunks")

client.create_collection(
    collection_name="chunks",
    vectors_config=VectorParams(size=embeddings.shape[1], distance=Distance.COSINE)
)

# 5. Upload points — each point carries its vector AND its metadata payload
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={
            "chunk_id":        row["chunk_id"],
            "company":         row["company"],
            "quarter":         row["quarter"],
            "year":            int(row["year"]),
            "company_quarter": row["company_quarter"],
            "start_turn_id":   int(row["start_turn_id"]),
            "end_turn_id":     int(row["end_turn_id"]),
            "speakers_in_chunk": row["speakers_in_chunk"],
            "chunk_text":      row["chunk_text"]
        }
    )
    for i, row in chunks_df.iterrows()
]

client.upsert(collection_name="chunks", points=points)

print(f"Indexed {len(points)} chunks into Qdrant")

.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4883.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 6/6 [00:14<00:00,  2.48s/it]


Indexed 192 chunks into Qdrant


## Creating and Testing retrieve variant

In [6]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

def retrieve_test(query, model, client, top_k=5, filter_company=None, filter_quarter=None):
    # Embed the query
    q_emb = model.encode([query], convert_to_numpy=True)[0].tolist()

    # Build metadata filter if provided
    conditions = []
    if filter_company:
        conditions.append(FieldCondition(key="company", match=MatchValue(value=filter_company)))
    if filter_quarter:
        conditions.append(FieldCondition(key="quarter", match=MatchValue(value=filter_quarter)))

    search_filter = Filter(must=conditions) if conditions else None

    # Search — filter applied inside Qdrant, no over-fetching needed
    results = client.query_points(
        collection_name="chunks",
        query=q_emb,
        query_filter=search_filter,
        limit=top_k,
        with_payload=True
    )

    return [
        {
            "chunk_id":   r.payload["chunk_id"],
            "company":    r.payload["company"],
            "quarter":    r.payload["quarter"],
            "score":      r.score,
            "chunk_text": r.payload["chunk_text"]
        }
        for r in results.points
    ]

In [7]:
results = retrieve_test(
    query="What were the key highlights from this earnings call?",
    model=model,
    client=client,
    top_k=5,
    filter_company="amazon",
    filter_quarter="Q3"
)

for r in results:
    print(r["chunk_id"], "| score:", round(r["score"], 3))
    print(r["chunk_text"][:200])
    print("---")

amazon_Q3_2025_chunk_0 | score: 0.488
[Speaker: Dave Fildes] Hello, and welcome to our Q3 2025 financial results conference call. Joining us today to answer your questions is Andy Jassy, our CEO; and Brian Olsavsky, our CFO. As you listen
---
amazon_Q3_2025_chunk_1 | score: 0.327
[Speaker: Brian Olsavsky] Thanks, Andy. Starting with our top line financial results. Worldwide revenue was $180.2 billion, a 12% increase year-over-year, excluding a 90 basis point favorable impact o
---
amazon_Q3_2025_chunk_11 | score: 0.202
[Speaker: Colin Sebastian] I guess first on AWS, following up there. How much of this acceleration is driven by core infrastructure versus AI workload monetization? And I think part of it is trying to
---
amazon_Q3_2025_chunk_6 | score: 0.169
[Speaker: Operator] The next question comes from the line of Mark Mahaney with Evercore ISI.
[Speaker: Mark Stephen Mahaney] I want to ask about 2 topics, groceries and then how to think about head co
---
amazon_Q3_2025_chunk_7 | sc